# Notebook 56 — SPECTER2 Embeddings on Merged Data

**Hypothesis**: SPECTER2 (allenai/specter2_base + classification adapter) produces richer scientific
paper representations than SPECTER1, boosting F1/AUC on the merged-training → AUB-test scenario.

**Prior art**:
- `nb54`: SPECTER1 on AUB-only → best config F1=0.5302, AUC=0.8257.
- `nb55`: SPECTER1 on merged data → Config C (LGBM tuned) best result on AUB test.
- REF-CLEAN (TF-IDF, AUB-only): F1=0.5109, AUC=0.8216.

**Approach**: Mirror nb55 exactly — same splits, same numeric features, same LGBM configs —
with SPECTER2 embeddings substituted for SPECTER1.

**SPECTER2 differences from SPECTER1**:
- Uses task-specific adapters (`allenai/specter2_base` + `allenai/specter2_classification`)
- CLS-token pooling instead of sentence-transformers `.encode()`
- Same 768-d output → fully drop-in compatible with downstream LGBM

**Configs**:

| Config | Text | Numeric | Classifier | Train data | Test data |
|--------|------|---------|------------|------------|-----------|
| REF-CLEAN | TF-IDF 5k | yes | LR | AUB-only | AUB-only |
| nb54-E | SPECTER1 768-d | yes | LGBM tuned | AUB-only | AUB-only |
| nb55-C | SPECTER1 768-d | yes | LGBM tuned | merged | AUB-only |
| A | SPECTER2 768-d | yes | LR | merged | AUB-only |
| B | SPECTER2 768-d | yes | LGBM | merged | AUB-only |
| C | SPECTER2 768-d | yes | LGBM tuned | merged | AUB-only |
| D | SPECTER2 768-d | yes | LGBM tuned | merged | all-unis |
| E | none | yes | LGBM | merged | AUB-only |


In [ ]:
import sys
sys.path.append('../../')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
from pathlib import Path
import copy
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, cohen_kappa_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from lightgbm import LGBMClassifier

RANDOM_STATE = 42
TRAIN_YEARS  = list(range(2010, 2018))
TEST_YEARS   = [2018, 2019, 2020]
QUANTILE     = 0.75

# Reference values
NB54_REF_CLEAN_F1  = 0.5109
NB54_REF_CLEAN_AUC = 0.8216
NB54_BEST_F1       = 0.5302   # nb54 Config E
NB54_BEST_AUC      = 0.8257
NB51_CLEAN_F1      = 0.5128
NB51_CLEAN_AUC     = 0.6654

CACHE_DIR = Path('../../data/cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print('Libraries loaded')

## 1. Load data

In [ ]:
data_path = Path('../../data/processed/all_unis_cleaned.pkl')

if not data_path.exists():
    raise FileNotFoundError(
        f"Merged data not found: {data_path}\n"
        "Run notebooks 04 → 05 → 06 first to generate the merged dataset."
    )

df = pd.read_pickle(data_path)

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"\nInstitution counts:")
print(df['institution'].value_counts().to_string())
print(f"\nYear range: {df['Year'].min()} – {df['Year'].max()}")

TITLE_COL = None
for candidate in ['Title', 'title', 'Document title', 'Paper Title', 'paper_title']:
    if candidate in df.columns:
        TITLE_COL = candidate
        break
print(f"\nTitle column: {TITLE_COL!r}")
print(f"Abstract column: 'Abstract' present = {'Abstract' in df.columns}")

In [ ]:
df_train     = df[df['Year'].isin(TRAIN_YEARS)].copy()
df_test_all  = df[df['Year'].isin(TEST_YEARS)].copy()
df_test_aub  = df_test_all[df_test_all['institution'] == 'AUB'].copy()
df_aub_train = df_train[df_train['institution'] == 'AUB'].copy()

print("SPLIT SUMMARY")
print("=" * 50)
print(f"Merged train (2010-2017): {len(df_train):,}")
print(df_train['institution'].value_counts().to_string())
print(f"\nAUB-only train (2010-2017): {len(df_aub_train):,}")
print(f"\nTest – all unis (2018-2020): {len(df_test_all):,}")
print(df_test_all['institution'].value_counts().to_string())
print(f"\nTest – AUB-only (2018-2020): {len(df_test_aub):,}")

## 2. Targets

In [ ]:
thr = df_aub_train['Citations'].quantile(QUANTILE)

y_train_merged = (df_train['Citations']     >= thr).astype(int)
y_train_aub    = (df_aub_train['Citations'] >= thr).astype(int)
y_test_aub     = (df_test_aub['Citations']  >= thr).astype(int)
y_test_all     = (df_test_all['Citations']  >= thr).astype(int)

print(f"Global threshold (AUB 75th pct): {thr:.0f} citations")
print(f"Merged train positive rate: {y_train_merged.mean():.1%}")
print(f"AUB    train positive rate: {y_train_aub.mean():.1%}")
print(f"AUB    test  positive rate: {y_test_aub.mean():.1%}")
print(f"All    test  positive rate: {y_test_all.mean():.1%}")

## 3. Feature helpers

In [ ]:
COL_MAP = {
    'snip':             'SNIP (publication year)',
    'snip_pct':         'SNIP percentile',
    'citescore':        'CiteScore (publication year)',
    'citescore_pct':    'CiteScore percentile',
    'sjr':              'SJR (publication year)',
    'sjr_pct':          'SJR percentile',
    'topic_prom':       'Topic Prominence Percentile',
    'num_authors':      'Authors',
    'num_institutions': 'Affiliations',
    'num_countries':    'Countries',
}


def extract_numeric_features(subset_df):
    vf = pd.DataFrame(index=subset_df.index)
    for feat, col in COL_MAP.items():
        if col in subset_df.columns:
            if col in ('Authors', 'Affiliations', 'Countries'):
                vf[feat] = subset_df[col].fillna('').apply(
                    lambda x: len(str(x).split(';')) if x else 1
                )
            else:
                vf[feat] = pd.to_numeric(subset_df[col], errors='coerce')
    return vf


def get_numeric_features(df_tr, df_te):
    vf_tr = extract_numeric_features(df_tr)
    vf_te = extract_numeric_features(df_te)
    tr_median = vf_tr.median()
    vf_tr = vf_tr.fillna(tr_median)
    vf_te = vf_te.fillna(tr_median)
    scaler = StandardScaler()
    X_tr_num = pd.DataFrame(scaler.fit_transform(vf_tr), index=vf_tr.index, columns=vf_tr.columns)
    X_te_num = pd.DataFrame(scaler.transform(vf_te),     index=vf_te.index, columns=vf_te.columns)
    return X_tr_num, X_te_num


def evaluate(model, X_tr, y_tr, X_te, y_te, label=''):
    """Fit model, grid-search threshold for best F1, return metrics dict."""
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_te)[:, 1]
    thresholds = np.arange(0.10, 0.91, 0.01)
    f1s    = [f1_score(y_te, (proba >= t).astype(int), zero_division=0) for t in thresholds]
    best_t = thresholds[int(np.argmax(f1s))]
    y_pred = (proba >= best_t).astype(int)
    return {
        'label':          label,
        'f1':             f1_score(y_te, y_pred, zero_division=0),
        'auc':            roc_auc_score(y_te, proba),
        'kappa':          cohen_kappa_score(y_te, y_pred),
        'recall':         recall_score(y_te, y_pred, zero_division=0),
        'precision':      precision_score(y_te, y_pred, zero_division=0),
        'threshold':      best_t,
        'n_train':        len(y_tr),
        'n_test':         len(y_te),
        'pos_rate_train': float(y_tr.mean()),
        'pos_rate_test':  float(y_te.mean()),
    }


print('Feature helpers ready.')

## 4. SPECTER2 embeddings

Uses `allenai/specter2_base` + `allenai/specter2_classification` adapter.
CLS-token pooling produces 768-d vectors — identical shape to SPECTER1.

In [ ]:
from transformers import AutoTokenizer
from adapters import AutoAdapterModel

SPECTER2_CACHE = CACHE_DIR / 'specter2_embeddings_merged.pkl'


def make_specter2_input(subset_df, title_col=None):
    """Build 'title [SEP] abstract' strings for SPECTER2 tokenizer."""
    abstracts = subset_df['Abstract'].fillna('').astype(str)
    if title_col and title_col in subset_df.columns:
        titles = subset_df[title_col].fillna('').astype(str)
        return (titles + ' [SEP] ' + abstracts).tolist()
    return abstracts.tolist()


def encode_specter2(tokenizer, model, texts, batch_size=32, device='cpu'):
    """Encode texts with SPECTER2, return (n, 768) numpy array."""
    model.eval()
    model.to(device)
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors='pt',
            max_length=512,
            return_token_type_ids=False,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        # CLS token (position 0) as the paper embedding
        cls_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_emb)
    return np.vstack(all_embeddings)


if SPECTER2_CACHE.exists():
    print('Loading cached SPECTER2 embeddings...')
    with open(SPECTER2_CACHE, 'rb') as f:
        cache = pickle.load(f)
    emb_train_merged = cache['train_merged']
    emb_train_aub    = cache['train_aub']
    emb_test_aub     = cache['test_aub']
    emb_test_all     = cache['test_all']
    print(f'  Merged train: {emb_train_merged.shape}')
    print(f'  AUB    train: {emb_train_aub.shape}')
    print(f'  AUB    test:  {emb_test_aub.shape}')
    print(f'  All    test:  {emb_test_all.shape}')
else:
    print('Loading SPECTER2 model + classification adapter...')
    sp2_tokenizer = AutoTokenizer.from_pretrained('allenai/specter2_base')
    sp2_model     = AutoAdapterModel.from_pretrained('allenai/specter2_base')
    sp2_model.load_adapter(
        'allenai/specter2_classification',
        source='hf',
        load_as='classification',
        set_active=True,
    )
    print(f'Model on {DEVICE}')

    splits = [
        ('train_merged', df_train),
        ('train_aub',    df_aub_train),
        ('test_aub',     df_test_aub),
        ('test_all',     df_test_all),
    ]

    cache = {}
    for name, subset in splits:
        texts = make_specter2_input(subset, TITLE_COL)
        print(f'  Encoding {len(texts):,} {name} papers...')
        cache[name] = encode_specter2(sp2_tokenizer, sp2_model, texts, device=DEVICE)
        print(f'    → {cache[name].shape}')

    emb_train_merged = cache['train_merged']
    emb_train_aub    = cache['train_aub']
    emb_test_aub     = cache['test_aub']
    emb_test_all     = cache['test_all']

    with open(SPECTER2_CACHE, 'wb') as f:
        pickle.dump(cache, f)
    print(f'Cached to {SPECTER2_CACHE}')

specter_cols = [f'sp_{i}' for i in range(emb_train_merged.shape[1])]

df_sp_train_merged = pd.DataFrame(emb_train_merged, index=df_train.index,     columns=specter_cols)
df_sp_train_aub    = pd.DataFrame(emb_train_aub,    index=df_aub_train.index, columns=specter_cols)
df_sp_test_aub     = pd.DataFrame(emb_test_aub,     index=df_test_aub.index,  columns=specter_cols)
df_sp_test_all     = pd.DataFrame(emb_test_all,     index=df_test_all.index,  columns=specter_cols)

print('\nSPECTER2 embeddings ready.')

## 5. Build feature matrices

In [ ]:
# --- Merged train → AUB test (Scenario A) ---
sp_scaler_merged = StandardScaler()
sp_tr_merged_scaled = pd.DataFrame(
    sp_scaler_merged.fit_transform(df_sp_train_merged),
    index=df_sp_train_merged.index, columns=specter_cols
)
sp_te_aub_scaled_merged = pd.DataFrame(
    sp_scaler_merged.transform(df_sp_test_aub),
    index=df_sp_test_aub.index, columns=specter_cols
)
sp_te_all_scaled_merged = pd.DataFrame(
    sp_scaler_merged.transform(df_sp_test_all),
    index=df_sp_test_all.index, columns=specter_cols
)

num_tr_merged, num_te_aub_from_merged = get_numeric_features(df_train,     df_test_aub)
_,             num_te_all_from_merged = get_numeric_features(df_train,     df_test_all)

X_merged_tr     = pd.concat([sp_tr_merged_scaled,     num_tr_merged.set_index(sp_tr_merged_scaled.index)], axis=1)
X_merged_te_aub = pd.concat([sp_te_aub_scaled_merged, num_te_aub_from_merged.set_index(sp_te_aub_scaled_merged.index)], axis=1)
X_merged_te_all = pd.concat([sp_te_all_scaled_merged, num_te_all_from_merged.set_index(sp_te_all_scaled_merged.index)], axis=1)

# --- AUB-only train → AUB test (ref baseline) ---
sp_scaler_aub = StandardScaler()
sp_tr_aub_scaled = pd.DataFrame(
    sp_scaler_aub.fit_transform(df_sp_train_aub),
    index=df_sp_train_aub.index, columns=specter_cols
)
sp_te_aub_scaled_aub = pd.DataFrame(
    sp_scaler_aub.transform(df_sp_test_aub),
    index=df_sp_test_aub.index, columns=specter_cols
)
num_tr_aub, num_te_aub_from_aub = get_numeric_features(df_aub_train, df_test_aub)
X_aub_tr     = pd.concat([sp_tr_aub_scaled,     num_tr_aub.set_index(sp_tr_aub_scaled.index)], axis=1)
X_aub_te_aub = pd.concat([sp_te_aub_scaled_aub, num_te_aub_from_aub.set_index(sp_te_aub_scaled_aub.index)], axis=1)

print(f"Feature matrices ready:")
print(f"  Merged train → AUB  test: {X_merged_tr.shape}  →  {X_merged_te_aub.shape}")
print(f"  Merged train → all  test: {X_merged_tr.shape}  →  {X_merged_te_all.shape}")
print(f"  AUB-only train → AUB test: {X_aub_tr.shape}  →  {X_aub_te_aub.shape}")

## 6. Model definitions

In [ ]:
n_pos_merged = int(y_train_merged.sum())
n_neg_merged = int((y_train_merged == 0).sum())
scale_w_merged = n_neg_merged / n_pos_merged

lr_model = LogisticRegression(
    max_iter=1000, class_weight='balanced',
    C=1.0, solver='lbfgs', random_state=RANDOM_STATE
)

lgbm_model_merged = LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=31,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_w_merged,
    random_state=RANDOM_STATE, verbose=-1,
)

print(f"scale_pos_weight (merged): {scale_w_merged:.2f}")

## 7. Experiments

### 7a. Config A — SPECTER2 + numeric, LR, merged train → AUB test

In [ ]:
results = []

print('=' * 65)
print('Config A: SPECTER2 + numeric, LogisticRegression, merged train')
print('=' * 65)

res_a = evaluate(copy.deepcopy(lr_model),
                 X_merged_tr.values, y_train_merged,
                 X_merged_te_aub.values, y_test_aub,
                 label='Config A (SPECTER2+num, LR, merged→AUB)')
results.append(res_a)

delta_f1  = res_a['f1']  - NB54_REF_CLEAN_F1
delta_auc = res_a['auc'] - NB54_REF_CLEAN_AUC
print(f"  F1:    {res_a['f1']:.4f}  ({delta_f1:+.4f} vs REF-CLEAN)")
print(f"  AUC:   {res_a['auc']:.4f}  ({delta_auc:+.4f} vs REF-CLEAN)")
print(f"  Kappa: {res_a['kappa']:.4f}")
print(f"  Recall: {res_a['recall']:.4f}  |  Precision: {res_a['precision']:.4f}")

### 7b. Config B — SPECTER2 + numeric, LGBM, merged train → AUB test

In [ ]:
print('=' * 65)
print('Config B: SPECTER2 + numeric, LGBM, merged train → AUB test')
print('=' * 65)

res_b = evaluate(copy.deepcopy(lgbm_model_merged),
                 X_merged_tr.values, y_train_merged,
                 X_merged_te_aub.values, y_test_aub,
                 label='Config B (SPECTER2+num, LGBM, merged→AUB)')
results.append(res_b)

delta_f1  = res_b['f1']  - NB54_REF_CLEAN_F1
delta_auc = res_b['auc'] - NB54_REF_CLEAN_AUC
print(f"  F1:    {res_b['f1']:.4f}  ({delta_f1:+.4f} vs REF-CLEAN)")
print(f"  AUC:   {res_b['auc']:.4f}  ({delta_auc:+.4f} vs REF-CLEAN)")
print(f"  Kappa: {res_b['kappa']:.4f}")
print(f"  Recall: {res_b['recall']:.4f}  |  Precision: {res_b['precision']:.4f}")

### 7c. Config C — SPECTER2 + numeric, LGBM tuned, merged train → AUB test

In [ ]:
print('=' * 65)
print('Config C: SPECTER2 + numeric, LGBM tuned, merged → AUB test')
print('=' * 65)

param_dist = {
    'n_estimators':      [200, 300, 500, 700],
    'learning_rate':     [0.01, 0.03, 0.05, 0.1],
    'num_leaves':        [15, 31, 63, 127],
    'min_child_samples': [10, 20, 30, 50],
    'subsample':         [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree':  [0.6, 0.7, 0.8, 0.9],
    'reg_alpha':         [0.0, 0.1, 0.5, 1.0],
    'reg_lambda':        [0.0, 0.1, 0.5, 1.0],
}

base_lgbm_merged = LGBMClassifier(
    scale_pos_weight=scale_w_merged,
    random_state=RANDOM_STATE, verbose=-1,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
search = RandomizedSearchCV(
    base_lgbm_merged, param_dist,
    n_iter=50, scoring='f1', cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=0
)
search.fit(X_merged_tr.values, y_train_merged)

print(f"  Best CV F1: {search.best_score_:.4f}")
print(f"  Best params: {search.best_params_}")

best_lgbm_merged = search.best_estimator_
res_c = evaluate(best_lgbm_merged,
                 X_merged_tr.values, y_train_merged,
                 X_merged_te_aub.values, y_test_aub,
                 label='Config C (SPECTER2+num, LGBM tuned, merged→AUB)')
results.append(res_c)

delta_f1  = res_c['f1']  - NB54_REF_CLEAN_F1
delta_auc = res_c['auc'] - NB54_REF_CLEAN_AUC
print(f"  F1:    {res_c['f1']:.4f}  ({delta_f1:+.4f} vs REF-CLEAN)  nb54-E was {NB54_BEST_F1:.4f}")
print(f"  AUC:   {res_c['auc']:.4f}  ({delta_auc:+.4f} vs REF-CLEAN)  nb54-E was {NB54_BEST_AUC:.4f}")
print(f"  Kappa: {res_c['kappa']:.4f}")
print(f"  Recall: {res_c['recall']:.4f}  |  Precision: {res_c['precision']:.4f}")

### 7d. Config D — SPECTER2 + numeric, LGBM tuned, merged → all-inst test (Scenario B)

In [ ]:
print('=' * 65)
print('Config D: SPECTER2 + numeric, LGBM tuned, merged → all-inst')
print('=' * 65)

res_d = evaluate(best_lgbm_merged,
                 X_merged_tr.values, y_train_merged,
                 X_merged_te_all.values, y_test_all,
                 label='Config D (SPECTER2+num, LGBM tuned, merged→all)')
results.append(res_d)

delta_f1  = res_d['f1']  - NB54_REF_CLEAN_F1
delta_auc = res_d['auc'] - NB54_REF_CLEAN_AUC
print(f"  F1:    {res_d['f1']:.4f}  ({delta_f1:+.4f} vs REF-CLEAN)")
print(f"  AUC:   {res_d['auc']:.4f}  ({delta_auc:+.4f} vs REF-CLEAN)")
print(f"  Kappa: {res_d['kappa']:.4f}")
print(f"  Recall: {res_d['recall']:.4f}  |  Precision: {res_d['precision']:.4f}")

### 7e. Config E — Numeric-only, LGBM, merged train → AUB test (ablation)

In [ ]:
print('=' * 65)
print('Config E: Numeric-only, LGBM, merged → AUB test (ablation)')
print('=' * 65)

res_e = evaluate(copy.deepcopy(lgbm_model_merged),
                 num_tr_merged.values, y_train_merged,
                 num_te_aub_from_merged.values, y_test_aub,
                 label='Config E (numeric-only, LGBM, merged→AUB)')
results.append(res_e)

delta_f1  = res_e['f1']  - NB54_REF_CLEAN_F1
delta_auc = res_e['auc'] - NB54_REF_CLEAN_AUC
print(f"  F1:    {res_e['f1']:.4f}  ({delta_f1:+.4f} vs REF-CLEAN)")
print(f"  AUC:   {res_e['auc']:.4f}  ({delta_auc:+.4f} vs REF-CLEAN)")
print(f"  Kappa: {res_e['kappa']:.4f}")
print(f"  Recall: {res_e['recall']:.4f}  |  Precision: {res_e['precision']:.4f}")

## 8. Results summary

In [ ]:
import matplotlib.pyplot as plt

ref_rows = [
    {'label': 'REF-CLEAN (TF-IDF+num, LR, AUB→AUB)',
     'f1': NB54_REF_CLEAN_F1, 'auc': NB54_REF_CLEAN_AUC, 'kappa': None,
     'recall': None, 'precision': None, 'threshold': None,
     'n_train': None, 'n_test': None,
     'pos_rate_train': None, 'pos_rate_test': None},
    {'label': 'nb54-E (SPECTER1+num, LGBM tuned, AUB→AUB)',
     'f1': NB54_BEST_F1, 'auc': NB54_BEST_AUC, 'kappa': None,
     'recall': None, 'precision': None, 'threshold': None,
     'n_train': None, 'n_test': None,
     'pos_rate_train': None, 'pos_rate_test': None},
]

res_df = pd.DataFrame(ref_rows + results)
res_df['delta_f1']  = res_df['f1']  - NB54_REF_CLEAN_F1
res_df['delta_auc'] = res_df['auc'] - NB54_REF_CLEAN_AUC

print('\n' + '=' * 115)
print('RESULTS SUMMARY — Notebook 56: SPECTER2 on Merged Data')
print('=' * 115)
cols = ['label', 'f1', 'delta_f1', 'auc', 'delta_auc', 'kappa']
print(res_df[cols].to_string(index=False, float_format='{:.4f}'.format))
print(f"\nSupervisor target: F1=0.7500")
print(f"Gap to target:     {0.75 - res_df['f1'].max():+.4f}")

new_best = res_df.loc[res_df['f1'].idxmax()]
print(f"\nBest config: {new_best['label']}")
print(f"  F1:    {new_best['f1']:.4f}  ({new_best['delta_f1']:+.4f} vs REF-CLEAN)")
print(f"  AUC:   {new_best['auc']:.4f}  ({new_best['delta_auc']:+.4f} vs REF-CLEAN)")
if pd.notna(new_best['kappa']):
    print(f"  Kappa: {new_best['kappa']:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

colors = ['#888888' if l.startswith('REF') or l.startswith('nb54') else
          ('#4C8BE2' if d >= 0 else '#E24C4C')
          for l, d in zip(res_df['label'], res_df['delta_f1'])]
labels_short = [
    l.split('(')[0].strip() if '(' in l else l
    for l in res_df['label']
]

# F1
axes[0].barh(labels_short, res_df['f1'], color=colors)
axes[0].axvline(NB51_CLEAN_F1, color='red',   linestyle='--', linewidth=1.5, label='nb51 baseline')
axes[0].axvline(0.75,          color='green', linestyle=':',  linewidth=1.5, label='Target 0.75')
axes[0].set_title('F1 Score')
axes[0].legend(fontsize=8)
for i, (v, d) in enumerate(zip(res_df['f1'], res_df['delta_f1'])):
    axes[0].text(v + 0.002, i, f'{v:.4f} ({d:+.4f})', va='center', fontsize=8)

# AUC
axes[1].barh(labels_short, res_df['auc'], color=colors)
axes[1].axvline(NB51_CLEAN_AUC, color='red', linestyle='--', linewidth=1.5, label='nb51 baseline')
axes[1].set_title('ROC-AUC')
axes[1].legend(fontsize=8)
for i, (v, d) in enumerate(zip(res_df['auc'], res_df['delta_auc'])):
    axes[1].text(v + 0.002, i, f'{v:.4f} ({d:+.4f})', va='center', fontsize=8)

# Kappa (only rows with values)
kappa_mask = res_df['kappa'].notna()
axes[2].barh(
    [labels_short[i] for i in res_df[kappa_mask].index],
    res_df[kappa_mask]['kappa'],
    color=[colors[i] for i in res_df[kappa_mask].index]
)
axes[2].axvline(0.4, color='orange', linestyle='--', linewidth=1.5, label='Moderate (0.4)')
axes[2].axvline(0.6, color='green',  linestyle='--', linewidth=1.5, label='Substantial (0.6)')
axes[2].set_title("Cohen's Kappa")
axes[2].legend(fontsize=8)
for i, v in zip(res_df[kappa_mask].index, res_df[kappa_mask]['kappa']):
    axes[2].text(v + 0.002, list(res_df[kappa_mask].index).index(i), f'{v:.4f}', va='center', fontsize=8)

plt.suptitle('Notebook 56 — SPECTER2 on Merged Data', fontsize=13, fontweight='bold')
plt.tight_layout()

docs_dir = Path('../../docs')
docs_dir.mkdir(exist_ok=True)
plt.savefig(docs_dir / 'nb56_specter2_merged_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

## 9. SPECTER1 vs SPECTER2 direct comparison

In [ ]:
# nb55 Config C results (SPECTER1 baseline for direct comparison)
NB55_C_F1  = None  # fill in after running nb55
NB55_C_AUC = None

print("=" * 70)
print("SPECTER1 vs SPECTER2 — Config C (LGBM tuned, merged→AUB)")
print("=" * 70)
print(f"  SPECTER1 (nb55-C):  F1={NB55_C_F1}  AUC={NB55_C_AUC}")
print(f"  SPECTER2 (nb56-C):  F1={res_c['f1']:.4f}  AUC={res_c['auc']:.4f}  Kappa={res_c['kappa']:.4f}")
if NB55_C_F1:
    delta_f1  = res_c['f1']  - NB55_C_F1
    delta_auc = res_c['auc'] - NB55_C_AUC
    print(f"  SPECTER2 vs SPECTER1:  ΔF1={delta_f1:+.4f}  ΔAUC={delta_auc:+.4f}")
    verdict = 'IMPROVES' if delta_f1 > 0.005 else ('HURTS' if delta_f1 < -0.005 else 'FLAT')
    print(f"  Verdict: SPECTER2 {verdict} over SPECTER1")
else:
    print("  (Run nb55 first and fill in NB55_C_F1 / NB55_C_AUC to see delta)")

## 10. Per-institution breakdown (Config D — all-inst test)

In [ ]:
proba_all = best_lgbm_merged.predict_proba(X_merged_te_all.values)[:, 1]
best_t_d  = res_d['threshold']

print(f"Per-institution breakdown — Config D (LGBM tuned, merged→all), threshold={best_t_d:.2f}")
print("=" * 75)
print(f"{'Institution':<15} {'n':>6} {'F1':>8} {'AUC':>8} {'Kappa':>8} {'Recall':>8} {'Precision':>10}")
print("-" * 75)

for inst in sorted(df_test_all['institution'].unique()):
    mask = df_test_all['institution'] == inst
    if mask.sum() < 10:
        continue
    y_i = y_test_all[mask]
    p_i = proba_all[mask.values]
    y_pred_i = (p_i >= best_t_d).astype(int)
    try:
        auc_i   = roc_auc_score(y_i, p_i)
        kappa_i = cohen_kappa_score(y_i, y_pred_i)
    except Exception:
        auc_i = kappa_i = float('nan')
    f1_i  = f1_score(y_i, y_pred_i, zero_division=0)
    rec_i = recall_score(y_i, y_pred_i, zero_division=0)
    pre_i = precision_score(y_i, y_pred_i, zero_division=0)
    marker = " ←" if inst == 'AUB' else ""
    print(f"{inst:<15} {mask.sum():>6,}  {f1_i:.4f}  {auc_i:.4f}  {kappa_i:.4f}  {rec_i:.4f}   {pre_i:.4f}{marker}")

## 11. Conclusions

In [ ]:
print('=' * 70)
print('NOTEBOOK 56 — CONCLUSIONS')
print('=' * 70)

best_nb56 = max(results, key=lambda x: x['f1'])

print(f"\nREF-CLEAN (TF-IDF, AUB-only):")
print(f"  F1={NB54_REF_CLEAN_F1:.4f}  AUC={NB54_REF_CLEAN_AUC:.4f}")

print(f"\nnb54-E (SPECTER1 LGBM tuned, AUB-only):")
print(f"  F1={NB54_BEST_F1:.4f}  AUC={NB54_BEST_AUC:.4f}")

print(f"\nBest nb56 config (SPECTER2): {best_nb56['label']}")
print(f"  F1:    {best_nb56['f1']:.4f}  ({best_nb56['f1'] - NB54_REF_CLEAN_F1:+.4f} vs REF-CLEAN)")
print(f"  AUC:   {best_nb56['auc']:.4f}  ({best_nb56['auc'] - NB54_REF_CLEAN_AUC:+.4f} vs REF-CLEAN)")
print(f"  Kappa: {best_nb56['kappa']:.4f}")
print(f"  Recall: {best_nb56['recall']:.4f}  |  Precision: {best_nb56['precision']:.4f}")

print(f"\nSupervisor target: F1=0.7500")
print(f"Gap to target:     {0.75 - best_nb56['f1']:+.4f}")

print('\n--- Full config summary ---')
for row in ref_rows:
    delta = row['f1'] - NB54_REF_CLEAN_F1
    sign  = '+' if delta >= 0 else ''
    print(f"  {'REFERENCE':9s}  {row['label']:55s}  F1={row['f1']:.4f}({sign}{delta:.4f})  AUC={row['auc']:.4f}")

for r in results:
    delta_f1  = r['f1']  - NB54_REF_CLEAN_F1
    delta_auc = r['auc'] - NB54_REF_CLEAN_AUC
    sign   = '+' if delta_f1 >= 0 else ''
    outcome = 'IMPROVED' if delta_f1 > 0.01 else ('DEGRADED' if delta_f1 < -0.01 else 'FLAT')
    kappa_str = f"  Kappa={r['kappa']:.4f}" if pd.notna(r['kappa']) else ''
    print(f"  {outcome:9s}  {r['label']:55s}  F1={r['f1']:.4f}({sign}{delta_f1:.4f})  AUC={r['auc']:.4f}({sign}{delta_auc:.4f}){kappa_str}")